# Level 3 (Hard): Employee Salary Analysis 📊💼

This notebook contains the complete Exploratory Data Analysis, Preprocessing, Statistical Modeling, and HR Analytics pipeline for employee compensation data.
Developed as part of the **Data Science Internship** at **Aadyam Talent Consultancy (ATC)**.

### **Objectives:**
1. **Data Preprocessing:** Standardize department columns, drop duplicates, handle negative compensation values, and impute missing experience fields based on education graduation ages.
2. **Descriptive Statistics:** Analyze overall distribution spread, headcount, and skewness of employee age, experience, and salaries.
3. **Statistical Modeling:** Run a Simple Linear Regression model to fit and evaluate the impact of experience on compensation.
4. **Visualizations & Demographics:** Map pay rates by department, education level, and check for gender pay equity.

In [ ]:
# Imports and Configuration
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Set custom plotting styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['grid.alpha'] = 0.3

## 1. Inspect Raw HR Dataset

In [ ]:
# Load raw dataset containing HR data anomalies
raw_df = pd.read_csv('dataset/raw_employee_salaries.csv')
print(f"Raw dataset shape: {raw_df.shape}")
print(f"Duplicate rows count: {raw_df.duplicated().sum()}")
print(f"Missing values in Experience: {raw_df['Years of Experience'].isna().sum()}")
print(f"Negative experience values: {(raw_df['Years of Experience'] < 0).sum()}")
print(f"Negative salary values: {(raw_df['Salary'] < 0).sum()}")
raw_df.head(3)

## 2. Execute Preprocessing & Data Cleaning

In [ ]:
from src.data_cleaner import clean_salary_data

# Run cleaning routine
df = clean_salary_data('dataset/raw_employee_salaries.csv', 'dataset/cleaned_employee_salaries.csv')
print(f"\nCleaned dataset shape: {df.shape}")

## 3. Statistical Demographics & Linear Regression

In [ ]:
# Filter out high-level outliers (e.g. CEO packages > $500k) to prevent OLS skewness
df_model = df[df['Is_Outlier'] == False]

# Descriptive Stats
print("=== DESCRIPTIVE STATISTICS ===")
print(df_model[['Age', 'Years of Experience', 'Salary']].describe())
print(f"\nSalary Distribution Skewness: {df_model['Salary'].skew():.4f}")

In [ ]:
# Fit Linear Regression: Salary = Intercept + (Slope * Experience)
X = df_model[['Years of Experience']].values
y = df_model['Salary'].values

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

print("=== LINEAR REGRESSION MODEL METRICS ===")
print(f"Formula            : Salary = {model.intercept_:.2f} + ({model.coef_[0]:.2f} * Experience)")
print(f"R-squared (R2)     : {r2_score(y, y_pred):.4f}")
print(f"RMSE ($)           : {np.sqrt(mean_squared_error(y, y_pred)):,.2f}")
print(f"Annual Raise Coefficient ($): {model.coef_[0]:,.2f} per year")

## 4. Visualizing Compensation Trends

### A. Salary Distribution

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df_model['Salary'], kde=True, color='#1E3A8A', bins=25, alpha=0.7)
plt.axvline(df_model['Salary'].mean(), color='#D97706', linestyle='--', linewidth=2, label=f"Mean: ${df_model['Salary'].mean():,.0f}")
plt.axvline(df_model['Salary'].median(), color='#0F766E', linestyle=':', linewidth=2, label=f"Median: ${df_model['Salary'].median():,.0f}")
plt.title('Employee Salary Distribution', fontweight='bold')
plt.xlabel('Salary ($)')
plt.legend()
plt.show()

### B. Experience vs. Salary Regression Plot

In [ ]:
plt.figure(figsize=(9.5, 5.5))
sns.regplot(
    x='Years of Experience', 
    y='Salary', 
    data=df_model, 
    scatter_kws={'alpha': 0.4, 'color': '#1E3A8A', 's': 20},
    line_kws={'color': '#D97706', 'linewidth': 2.5, 'label': 'OLS Trendline'}
)
plt.title('Salary vs. Experience OLS Regression Plot', fontweight='bold')
plt.xlabel('Years of Experience')
plt.ylabel('Salary ($)')
plt.legend()
plt.show()

### C. Salary Spread by Education Level

In [ ]:
plt.figure(figsize=(8, 5))
edu_order = ['High School', "Bachelor's", "Master's", 'PhD']
sns.boxplot(
    x='Education Level', 
    y='Salary', 
    data=df_model, 
    order=edu_order,
    palette=['#4B5563', '#1E3A8A', '#0F766E', '#D97706']
)
plt.title('Salary Box Plot by Education Level', fontweight='bold')
plt.xlabel('Education Level')
plt.ylabel('Salary ($)')
plt.show()

### D. Department Comparison & Correlation matrix

In [ ]:
corr = df_model[['Age', 'Years of Experience', 'Salary', 'Performance Rating']].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='Blues', square=True, linewidths=0.5)
plt.title('Correlation Matrix Heatmap', fontweight='bold')
plt.show()

## 5. Summary Compensation Insights

1. **Strong Experience Effect:** Years of experience is the single strongest predictor of salary, with a correlation coefficient of `0.804`. Each year of experience is worth an estimated increase of `$3,960.54` in compensation.
2. **Education Premium:** Clear graduation salary steps exist (Master's averages `$112.3k`, PhD averages `$125.3k` over Bachelor's at `$107.3k`).
3. **Department Scales:** Executive leadership and Engineering represent the highest departmental pay caps, while HR represents the lowest baseline caps. This informs localized budgeting models.